# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from huggingface_hub import snapshot_download

# ---------------------------------------------------------
# 1. Hugging Face token
# ---------------------------------------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

# ---------------------------------------------------------
# 2. Download ONLY the months/tables we need
# ---------------------------------------------------------

warehouse_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        "fact_content_daily_performance/month=2026-03/*.parquet",
        "fact_content_daily_performance/month=2026-04/*.parquet",
        "dim_clients.parquet",
        "dim_content.parquet",
    ],
)

print("Warehouse files available locally.")
print("Path:", warehouse_path)

# ---------------------------------------------------------
# 3. Connect DuckDB
# ---------------------------------------------------------

con = duckdb.connect()

MARCH = (
    f"read_parquet("
    f"'{warehouse_path}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

APRIL = (
    f"read_parquet("
    f"'{warehouse_path}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

DIM_CLIENTS = (
    f"read_parquet("
    f"'{warehouse_path}/dim_clients.parquet'"
    f")"
)

DIM_CONTENT = (
    f"read_parquet("
    f"'{warehouse_path}/dim_content.parquet'"
    f")"
)

print("DuckDB connected.")

Note: you may need to restart the kernel to use updated packages.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Warehouse files available locally.
Path: /Users/mehdidardour/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
DuckDB connected.


In [5]:
test_read = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions
    FROM {MARCH}
    LIMIT 5
""").df()

test_read

,report_date,client_hash_id,content_hash_id,gsc_impressions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

- **Unit of analysis:** One row in the raw daily fact table represents one report date × one client × one content item. For this project, I aggregate daily observations into one row per content item for ranking.

- **Feature window:** March 1 to March 31, 2026.

- **Outcome window:** April 1 to April 30, 2026.

- **Prediction / ranking goal:** Rank content items by their risk of a meaningful next-month decline in search impressions.

- **Proxy label:** `is_future_decline = 1` when April impressions are less than 80% of March impressions.

- **Deliberately excluded:** April performance metrics are excluded from the feature set because they are future information and would create leakage.

In [6]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

I use five features:

1. `impressions_march` — total GSC impressions observed during March.
2. `clicks_march` — total GSC clicks observed during March.
3. `ctr_march` — March clicks divided by March impressions.
4. `avg_position_march` — average GSC search position during March.
5. `days_with_impressions` — number of March days with at least one impression.

### Label

`is_future_decline` — 1 if April impressions are less than 80% of March impressions, otherwise 0.

### Context

- `client_hash_id` — used only for grouping and validation.
- `content_hash_id` — identifies the content item.
- `report_date` — defines the feature and outcome windows.
- `gsc_data_available` — indicates whether GSC data is actually available.

### Excluded

- April performance metrics are excluded from the feature set because they are future information.
- `client_hash_id` and `content_hash_id` are not model features because the pseudonymous identifiers themselves have no meaningful predictive interpretation.
- The fixed 90-day query table is excluded from this first feature frame because its time window may overlap the outcome window and create leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Query 1 — Grain

The raw warehouse table should contain one row per report date × client × content item.  
If the query below returns no rows, the expected grain holds.

In [12]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

,report_date,client_hash_id,content_hash_id,row_count


### Verification Query 2 — Row count and date window

This query checks the number of rows in the March slice and confirms the observed date window.

In [13]:
march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH}
""").df()

march_summary

,rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Verification Query 3 — Data availability

This query checks how many March rows have available GSC data.  
I use `IS TRUE` so unavailable or unknown tracking is not treated as real zero activity.

In [14]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_rows
    FROM {MARCH}
""").df()

availability

,total_rows,available_rows
0,9841378,3611061


### Five-feature frame

The decision moment is the end of March 2026.  
I therefore build five simple features using only information observed during March.

In [15]:
data = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        100.0 * SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        AVG(gsc_avg_position) AS avg_position,

        COUNT(*) FILTER (
            WHERE gsc_impressions > 0
        ) AS active_days

    FROM {MARCH}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

data.head()

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,active_days
0,client_e547b89c05043229,content_5f531923e885f10f,2182.0,0.0,0.000000,22.396646,29
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.142017,2.563756,29
2,client_e547b89c05043229,content_e66540b8336f76bc,1160.0,0.0,0.000000,3.787335,29
3,client_e547b89c05043229,content_5b2a5a9ecd7aafc6,63.0,0.0,0.000000,17.785507,23
4,client_e547b89c05043229,content_f59c8de7c55b528a,2261.0,8.0,0.353826,6.749803,29


### Feature availability

- `impressions`: available at decision time because it only uses impressions observed during March.
- `clicks`: available at decision time because it only uses clicks observed during March.
- `ctr`: available at decision time because it is calculated only from March clicks and impressions.
- `avg_position`: available at decision time because it only uses search positions observed during March.
- `active_days`: available at decision time because it only counts March days with observed impressions.

### Future outcome

To create a simple future-looking proxy label, I compare March performance with April performance.

`is_future_decline = 1` when April impressions are more than 20% lower than March impressions.

In [16]:
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions

    FROM {APRIL}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

data = data.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

data["is_future_decline"] = (
    data["april_impressions"] < 0.8 * data["impressions"]
).astype(int)

data.head()

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,active_days,april_impressions,is_future_decline
0,client_e547b89c05043229,content_5f531923e885f10f,2182.0,0.0,0.000000,22.396646,29,548.0,1
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.142017,2.563756,29,191648.0,0
2,client_e547b89c05043229,content_e66540b8336f76bc,1160.0,0.0,0.000000,3.787335,29,1771.0,0
3,client_e547b89c05043229,content_5b2a5a9ecd7aafc6,63.0,0.0,0.000000,17.785507,23,68.0,0
4,client_e547b89c05043229,content_f59c8de7c55b528a,2261.0,8.0,0.353826,6.749803,29,2647.0,0


### Honest model

I first train a simple model using only the five March features.  
This gives me an honest score because no April information is used as a feature.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "active_days"
]

model_data = data.dropna(
    subset=features + ["is_future_decline"]
).copy()

X = model_data[features]
y = model_data["is_future_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

honest_score = accuracy_score(
    y_test,
    model.predict(X_test)
)

print("Honest score:", honest_score)

Honest score: 0.5874161158484282


### Deliberate leakage experiment

I now deliberately add a feature that directly contains the target information.  
This should make the score look artificially better and demonstrates target leakage.

In [18]:
# Deliberate leakage
model_data["label_leak"] = model_data["is_future_decline"]

leaky_features = features + ["label_leak"]

X = model_data[leaky_features]
y = model_data["is_future_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_score = accuracy_score(
    y_test,
    leaky_model.predict(X_test)
)

print("Honest score:", honest_score)
print("Leaky score:", leaky_score)

# Remove the leaked feature
model_data = model_data.drop(columns=["label_leak"])

print(
    "Leak removed:",
    "label_leak" not in model_data.columns
)

Honest score: 0.5874161158484282
Leaky score: 1.0
Leak removed: True


### Leakage lesson

The leaked score is artificially high because `label_leak` directly contains information from the target.

I removed this feature and keep the honest score as the valid result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

One limitation is that the warehouse is an unbalanced panel: different clients have different amounts of observed history.

Some rows can also exist before tracking was fully available, so missing or unavailable measurements should not automatically be interpreted as zero activity.

Finally, the future decline label is only a directional signal. It does not prove that refreshing a page would cause its performance to improve.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.